# University Waste Management & Agricultural Field Monitoring System
### HSV Green-Masking + YOLOv8 — Colab GPU Pipeline

This notebook runs the full pipeline end-to-end on a Colab GPU runtime:

1. Environment setup (GPU check, Drive mount, dependencies)
2. Dataset download (Kaggle) + sanity inspection
3. Dataset assembly: single-class merge, train/val/test split
4. HSV green-masking preprocessing (visual pipeline check)
5. Baseline (unmasked) vs HSV-masked training — the core ablation
6. Metrics comparison (mAP50, mAP50-95, precision, recall)
7. Inference on original RGB images with green-ratio post-filter
8. Persist everything to Google Drive

**Before running:** `Runtime > Change runtime type > T4 GPU` (or better).

## 1. Environment setup

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/university-waste-management'
import os
os.makedirs(PROJECT_ROOT, exist_ok=True)
print('Project root:', PROJECT_ROOT)

Clone the project repo (containing `src/`) into the Colab runtime. Replace `REPO_URL`
if you've pushed this project to GitHub — otherwise upload the `src/` folder manually via the
Colab file browser into `/content/University-Waste-Management/`.

In [ ]:
REPO_URL = ''  # e.g. 'https://github.com/<you>/university-waste-management.git'
CODE_DIR = '/content/University-Waste-Management'

if REPO_URL:
    !git clone -q {REPO_URL} {CODE_DIR}
else:
    os.makedirs(CODE_DIR, exist_ok=True)
    print('REPO_URL not set — upload/sync src/ into', CODE_DIR, 'manually before continuing.')

import sys
sys.path.insert(0, CODE_DIR)

In [ ]:
!pip install -q ultralytics opencv-python-headless kaggle pyyaml tqdm seaborn

## 2. Dataset download (Kaggle)

Upload your `kaggle.json` API token when prompted (Kaggle account → Settings → Create New API Token).

In [ ]:
from google.colab import files
import os

kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)

if not os.path.exists(f'{kaggle_dir}/kaggle.json'):
    uploaded = files.upload()  # select kaggle.json
    for fname in uploaded:
        os.rename(fname, f'{kaggle_dir}/kaggle.json')
    os.chmod(f'{kaggle_dir}/kaggle.json', 0o600)
print('Kaggle credentials ready.')

In [ ]:
RAW_DATA_DIR = f'{PROJECT_ROOT}/data/raw'
os.makedirs(RAW_DATA_DIR, exist_ok=True)

!kaggle datasets download -d ravirajsinh45/crop-and-weed-detection-data-with-bounding-boxes -p {RAW_DATA_DIR} --unzip
!find {RAW_DATA_DIR} -maxdepth 3 | head -30

## 3. Dataset assembly

Discover image/label pairs, **inspect the class distribution before trusting the 0/1 class-id convention**, merge crop+weed into a single `green_vegetation` class, and split train/val/test.

In [ ]:
from pathlib import Path
from src.preprocessing.dataset_prep import (
    discover_pairs, inspect_class_distribution, split_dataset, write_data_yaml
)

pairs = discover_pairs(Path(RAW_DATA_DIR))
print(f'Found {len(pairs)} image/label pairs')
print('Class distribution (verify before merging!):', inspect_class_distribution(pairs))

In [ ]:
BASELINE_DIR = Path(PROJECT_ROOT) / 'data' / 'baseline'
split_dataset(pairs, BASELINE_DIR, train=0.7, val=0.2, test=0.1, seed=42, merge_to_single_class=True)

baseline_yaml = write_data_yaml(BASELINE_DIR / 'data.yaml', BASELINE_DIR, names=['green_vegetation'])
print('Baseline data.yaml:', baseline_yaml)

## 4. HSV green-masking — visual sanity check

Inspect the mask and both masking strategies (hard black-out vs soft desaturation) on a few sample images before committing to a full-dataset pass.

In [ ]:
import cv2
import matplotlib.pyplot as plt
from src.preprocessing.hsv_mask import MaskConfig, visualize_pipeline

sample_paths = list((BASELINE_DIR / 'images' / 'train').iterdir())[:3]
mask_config = MaskConfig()  # tune DEFAULT_LOWER_GREEN/UPPER_GREEN in hsv_mask.py if needed

fig, axes = plt.subplots(len(sample_paths), 4, figsize=(16, 4 * len(sample_paths)))
for row, img_path in enumerate(sample_paths):
    image = cv2.imread(str(img_path))
    mask, hard, soft = visualize_pipeline(image, mask_config)
    for col, (title, img) in enumerate([
        ('original', image), ('mask', mask), ('hard-masked', hard), ('soft-masked', soft)
    ]):
        ax = axes[row, col] if len(sample_paths) > 1 else axes[col]
        cmap = 'gray' if img.ndim == 2 else None
        disp = img if img.ndim == 2 else cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(disp, cmap=cmap)
        ax.set_title(title)
        ax.axis('off')
fig.tight_layout()
plt.show()

**If the mask misses vegetation or catches soil/straw**, tune `DEFAULT_LOWER_GREEN` /
`DEFAULT_UPPER_GREEN` in `src/preprocessing/hsv_mask.py` (or pass a custom `MaskConfig` above)
and re-run this cell before generating the full masked dataset.

In [ ]:
from src.preprocessing.dataset_prep import build_masked_variant

MASKED_DIR = Path(PROJECT_ROOT) / 'data' / 'masked'
build_masked_variant(BASELINE_DIR, MASKED_DIR, mode='soft', config=mask_config)
masked_yaml = write_data_yaml(MASKED_DIR / 'data.yaml', MASKED_DIR, names=['green_vegetation'])
print('Masked data.yaml:', masked_yaml)

## 5. Train — baseline vs HSV-masked ablation

Same architecture, same hyperparameters, only the preprocessing differs. This is the core evidence for the project's thesis.

In [ ]:
from src.training.train import TrainConfig, train

RUNS_DIR = f'{PROJECT_ROOT}/runs'

baseline_cfg = TrainConfig(
    data_yaml=str(baseline_yaml),
    model='yolov8s.pt',
    epochs=100,
    imgsz=512,
    batch=16,
    project=RUNS_DIR,
    name='baseline',
)
baseline_model, baseline_results = train(baseline_cfg)

In [ ]:
masked_cfg = TrainConfig(
    data_yaml=str(masked_yaml),
    model='yolov8s.pt',
    epochs=100,
    imgsz=512,
    batch=16,
    project=RUNS_DIR,
    name='masked',
)
masked_model, masked_results = train(masked_cfg)

## 6. Compare metrics

In [ ]:
from src.evaluation.metrics import load_results_csv, plot_loss_curves, plot_map_curves, compare_runs

baseline_run_dir = f'{RUNS_DIR}/baseline'
masked_run_dir = f'{RUNS_DIR}/masked'

baseline_df = load_results_csv(baseline_run_dir)
masked_df = load_results_csv(masked_run_dir)

plot_loss_curves(baseline_df, f'{PROJECT_ROOT}/reports/baseline_loss.png', 'Baseline — Loss')
plot_loss_curves(masked_df, f'{PROJECT_ROOT}/reports/masked_loss.png', 'HSV-Masked — Loss')
plot_map_curves(baseline_df, f'{PROJECT_ROOT}/reports/baseline_map.png', 'Baseline — mAP')
plot_map_curves(masked_df, f'{PROJECT_ROOT}/reports/masked_map.png', 'HSV-Masked — mAP')

summary = compare_runs(baseline_run_dir, masked_run_dir, f'{PROJECT_ROOT}/reports/comparison.png')
summary

## 7. Inference on original RGB images

The masked model still runs on **unmasked** images at inference time — masking is a training-time noise filter only. The optional green-ratio post-filter drops boxes that don't actually contain green pixels.

In [ ]:
from src.inference.predict import predict_and_save

test_images = list((BASELINE_DIR / 'images' / 'test').iterdir())[:5]
masked_weights = f'{masked_run_dir}/weights/best.pt'

for img_path in test_images:
    out_path = f'{PROJECT_ROOT}/reports/inference/{img_path.stem}_pred.jpg'
    predict_and_save(
        masked_weights, img_path, out_path,
        conf=0.25, green_ratio_threshold=0.15, mask_config=mask_config,
    )
print('Saved annotated predictions to', f'{PROJECT_ROOT}/reports/inference/')

In [ ]:
import matplotlib.pyplot as plt
import cv2
from pathlib import Path

preds = sorted(Path(f'{PROJECT_ROOT}/reports/inference').glob('*_pred.jpg'))[:5]
fig, axes = plt.subplots(1, len(preds), figsize=(4 * len(preds), 4))
for ax, p in zip(axes, preds):
    ax.imshow(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB))
    ax.set_title(p.stem)
    ax.axis('off')
fig.tight_layout()
plt.show()

## 7b. Combined pipeline figure — before → mask → detection → leaf count

The report/competition-ready artifact: one figure per sample showing
**Original → Green Mask → Masked (training view) → Final Detection → Leaf
Count**, side by side, so the whole preprocessing + counting story is
visible at a glance. Leaf counting is a classical watershed split on the
mask (see `src/analysis/leaf_counter.py`) — an estimate, not ground truth;
it under-counts leaves that fully overlap in the 2D projection. Tune
`leaf_min_area` / `leaf_fg_ratio` below if counts look off for your images.

In [ ]:
from src.evaluation.report import plot_pipeline_grid

demo_images = list((BASELINE_DIR / 'images' / 'test').iterdir())[:5]

pipeline_fig_path = plot_pipeline_grid(
    demo_images,
    masked_weights,
    f'{PROJECT_ROOT}/reports/pipeline_demo.png',
    mask_config=mask_config,
    conf=0.25,
    green_ratio_threshold=0.15,
)
print('Saved pipeline figure to', pipeline_fig_path)

In [ ]:
import matplotlib.pyplot as plt
import cv2

fig_img = cv2.cvtColor(cv2.imread(str(pipeline_fig_path)), cv2.COLOR_BGR2RGB)
plt.figure(figsize=(16, 4 * len(demo_images)))
plt.imshow(fig_img)
plt.axis('off')
plt.show()

## 7c. Test on your own photos

Upload images straight from your device to sanity-check the trained model
on real-world shots outside the dataset's distribution (different framing,
lighting, distance) — including the leaf count. If detections come back
empty, try `conf=0.1` and `green_ratio_threshold=None` first to see raw
model confidence before the post-filters — that's a distribution-shift
finding worth reporting, not necessarily a bug.

In [ ]:
from google.colab import files
from pathlib import Path

upload_dir = Path(f'{PROJECT_ROOT}/data/custom_test')
upload_dir.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()  # pick photos from your device
custom_images = []
for fname, content in uploaded.items():
    dest = upload_dir / fname
    dest.write_bytes(content)
    custom_images.append(dest)

print(f'Uploaded {len(custom_images)} image(s) to {upload_dir}')

In [ ]:
custom_fig_path = plot_pipeline_grid(
    custom_images,
    masked_weights,
    f'{PROJECT_ROOT}/reports/custom_test_pipeline.png',
    mask_config=mask_config,
    conf=0.25,
    green_ratio_threshold=0.15,
)

import matplotlib.pyplot as plt, cv2
fig_img = cv2.cvtColor(cv2.imread(str(custom_fig_path)), cv2.COLOR_BGR2RGB)
plt.figure(figsize=(16, 4 * len(custom_images)))
plt.imshow(fig_img)
plt.axis('off')
plt.show()

## 8. Waste detection — second model

A separate, independent model for non-green waste categories (trash,
paper, plastic, soil anomalies, etc.) — deliberately **not** merged into
the vegetation model, so nothing here can change the baseline-vs-masked
ablation results you already have. Default target dataset: [TACO
(Trash Annotations in Context) — YOLO format](https://www.kaggle.com/datasets/vencerlanz09/taco-dataset-yolo-format),
real-scene litter photos with bounding boxes across dozens of fine-grained
categories.

**This dataset's exact category scheme needs verifying after download** —
same discipline as step 3. Run the inspection cell below, look at the
printed class names/ids, then fill in `WASTE_CLASS_MAP` before splitting.
Don't skip straight to training on an unverified guess.

In [ ]:
WASTE_RAW_DIR = f'{PROJECT_ROOT}/data/waste_raw'
import os
os.makedirs(WASTE_RAW_DIR, exist_ok=True)

!kaggle datasets download -d vencerlanz09/taco-dataset-yolo-format -p {WASTE_RAW_DIR} --unzip
!find {WASTE_RAW_DIR} -maxdepth 3 | head -30

In [ ]:
from src.preprocessing.dataset_prep import discover_pairs, inspect_class_distribution, find_class_names

waste_pairs = discover_pairs(Path(WASTE_RAW_DIR))
print(f'Found {len(waste_pairs)} image/label pairs')

waste_class_dist = inspect_class_distribution(waste_pairs)
waste_class_names_raw = find_class_names(Path(WASTE_RAW_DIR))
print('Class distribution:', waste_class_dist)
print('Discovered class names (id -> name, if a classes.txt/yaml was found):')
if waste_class_names_raw:
    for idx, name in enumerate(waste_class_names_raw):
        print(f'  {idx}: {name}  (count={waste_class_dist.get(idx, 0)})')
else:
    print('  No classes.txt/yaml found — cross-reference ids against the Kaggle dataset card manually.')

**Edit this before continuing.** `WASTE_CLASS_MAP` maps *source* class id ->
*output* class id, using the ids/names printed above. Any source id you
leave out of the map is dropped entirely (useful for categories with too
few examples to train on). The placeholder below groups into 5 practical
supercategories — adjust the keys to match what you actually saw printed,
and adjust `WASTE_CLASS_NAMES` to match the values you chose.

In [ ]:
# EDIT THESE TWO based on the class list printed above — this placeholder
# assumes a 0..N raw id scheme and will very likely need remapping.
WASTE_CLASS_NAMES = ['plastic', 'paper', 'metal_glass', 'organic', 'other_rubbish']
WASTE_CLASS_MAP = {
    # source_id: output_id
    0: 0, 1: 0,      # example: plastic bag, plastic bottle -> plastic
    2: 1, 3: 1,       # example: paper, cardboard -> paper
    4: 2, 5: 2,       # example: can, glass jar -> metal_glass
    6: 3,             # example: food waste -> organic
    7: 4,             # example: unlabeled litter -> other_rubbish
}
print('Using WASTE_CLASS_MAP:', WASTE_CLASS_MAP)

In [ ]:
from src.preprocessing.dataset_prep import split_dataset, write_data_yaml

WASTE_DIR = Path(PROJECT_ROOT) / 'data' / 'waste'
split_dataset(waste_pairs, WASTE_DIR, train=0.7, val=0.2, test=0.1, seed=42, class_id_map=WASTE_CLASS_MAP)
waste_yaml = write_data_yaml(WASTE_DIR / 'data.yaml', WASTE_DIR, names=WASTE_CLASS_NAMES)
print('Waste data.yaml:', waste_yaml)

Train the waste model — same wrapper as the vegetation model, just pointed
at a different `data.yaml`. A smaller/faster backbone (`yolov8n`) is a
reasonable default here since this is a secondary model, not the project's
headline ablation; bump to `yolov8s` if you have GPU time to spare.

In [ ]:
waste_cfg = TrainConfig(
    data_yaml=str(waste_yaml),
    model='yolov8n.pt',
    epochs=80,
    imgsz=512,
    batch=16,
    project=RUNS_DIR,
    name='waste',
)
waste_model, waste_results = train(waste_cfg)
waste_weights = f'{RUNS_DIR}/waste/weights/best.pt'

## 9. Combined inference — vegetation + waste together

Runs both models independently on the same original image and merges the
two detection sets into one annotated view: green boxes for vegetation,
red boxes for waste categories. This is the "complete scene" artifact —
one image, both models, nothing retrained or merged at the weights level.

In [ ]:
from src.inference.combined_predict import predict_combined_and_save

combined_demo_images = list((BASELINE_DIR / 'images' / 'test').iterdir())[:5]

for img_path in combined_demo_images:
    out_path = f'{PROJECT_ROOT}/reports/combined/{img_path.stem}_combined.jpg'
    predict_combined_and_save(
        img_path, masked_weights, waste_weights, out_path,
        veg_conf=0.25, waste_conf=0.25,
        veg_class_names=['green_vegetation'],
        waste_class_names=WASTE_CLASS_NAMES,
        veg_green_ratio_threshold=0.15,
        mask_config=mask_config,
    )
print('Saved combined annotations to', f'{PROJECT_ROOT}/reports/combined/')

In [ ]:
import matplotlib.pyplot as plt
import cv2

combined_preds = sorted(Path(f'{PROJECT_ROOT}/reports/combined').glob('*_combined.jpg'))
fig, axes = plt.subplots(1, len(combined_preds), figsize=(4 * len(combined_preds), 4))
for ax, p in zip(axes, combined_preds):
    ax.imshow(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB))
    ax.set_title(p.stem, fontsize=8)
    ax.axis('off')
fig.tight_layout()
plt.show()

## 10. Everything is already on Drive

Since `PROJECT_ROOT` lives under `/content/drive/MyDrive/...`, weights (`runs/*/weights/best.pt`), plots (`reports/`), and the assembled datasets all persist automatically across Colab sessions — no extra export step needed.

In [ ]:
print('Baseline weights:', f'{baseline_run_dir}/weights/best.pt')
print('Masked weights:  ', f'{masked_run_dir}/weights/best.pt')
print('Reports:         ', f'{PROJECT_ROOT}/reports/')
print()
print('Final metrics summary:')
summary
print('Waste weights:     ', f'{RUNS_DIR}/waste/weights/best.pt')